# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# =========================
# 1. Connect to internship repo
# =========================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")


# =========================
# 2. Hugging Face token
# =========================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")


# =========================
# 3. DuckDB connection
# =========================

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")


# =========================
# 4. Load March 2026 data
# =========================

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        sessions_ai,
        sessions_paid,
        sessions_direct,
        sessions_social,
        sessions_organic,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("March 2026 shape:", march_df.shape)

print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,sessions_paid,sessions_direct,sessions_social,sessions_organic,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

I first inspect the distributions of the key signals before defining any rule.

The signals are highly granular page-level observations, so I use simple descriptive statistics and buckets rather than assuming a normal distribution.

The purpose is to understand scale, missingness, and whether the signals have enough variation to support a decision-support rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "scroll_events"
]

distribution_table = (
    march_df[signal_cols]
    .describe()
    .T
)

display(distribution_table)

missing_table = (
    march_df[signal_cols]
    .isna()
    .sum()
    .reset_index()
)

missing_table.columns = ["signal", "missing_n"]

display(missing_table)

,count,mean,std,min,25%,50%,75%,max
gsc_impressions,9841378.0,28.518119,155.926569,0.0,0.0,0.0,6.0,40084.0
gsc_clicks,9841378.0,0.083508,0.781434,0.0,0.0,0.0,0.0,274.0
gsc_sum_position,9841378.0,329.87423,2360.105635,0.0,0.0,0.0,60.0,481946.0
scroll_events,6822637.0,0.032261,0.413166,0.0,0.0,0.0,0.0,254.0


,signal,missing_n
0,gsc_impressions,0
1,gsc_clicks,0
2,gsc_sum_position,0
3,scroll_events,3018741


The descriptive statistics show the observed scale and spread of the signals. Missing values are also important because a signal should not be interpreted as evidence when its source data is unavailable.

These distributions are descriptive only and are not evidence of causality.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 2. Signal test #1 — GSC impressions

GSC impressions are used as a search-visibility signal.

Hypothesis: pages with higher observed impressions have materially different search visibility from pages with lower impressions.

This is a safe decision-time signal because it describes observed performance rather than a future outcome.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal1 = march_df[["gsc_impressions"]].copy()

signal1["bucket"] = pd.qcut(
    signal1["gsc_impressions"],
    q=4,
    duplicates="drop"
)

signal1_table = (
    signal1.groupby("bucket", observed=True)
    .agg(
        n=("gsc_impressions", "size"),
        median=("gsc_impressions", "median"),
        mean=("gsc_impressions", "mean")
    )
    .reset_index()
)

display(signal1_table)

,bucket,n,median,mean
0,"(-0.001, 6.0]",7439673,0.0,0.448958
1,"(6.0, 40084.0]",2401705,38.0,115.466924


### Verdict: CONFIRMED

The observed buckets show clear variation in search impressions. This supports using impressions as a search-visibility signal in a decision-support rule.

The result is directional and does not establish that impressions cause future improvement.

## Signal test #2 — GSC clicks

GSC clicks are used as an observed search-engagement signal.

Hypothesis: pages with different click levels represent meaningfully different observed search engagement.

Clicks are measured at the decision moment and do not require a future outcome.

In [4]:
signal2 = march_df[["gsc_clicks"]].copy()

signal2["bucket"] = pd.qcut(
    signal2["gsc_clicks"],
    q=4,
    duplicates="drop"
)

signal2_table = (
    signal2.groupby("bucket", observed=True)
    .agg(
        n=("gsc_clicks", "size"),
        median=("gsc_clicks", "median"),
        mean=("gsc_clicks", "mean")
    )
    .reset_index()
)

display(signal2_table)

,bucket,n,median,mean
0,"(-0.001, 274.0]",9841378,0.0,0.083508


## Signal test #3 — GSC position

GSC sum position is used as the flag-linked search-position signal.

The session's CTR-fix logic uses CTR-vs-position, so search position is a relevant signal to audit.

Hypothesis: observed search position varies enough across pages to distinguish different search contexts.

Lower position values represent better observed search placement, while larger values represent weaker placement.

This test is descriptive and does not claim that position alone causes future performance.

In [5]:
signal3 = march_df[["gsc_sum_position"]].copy()

signal3["bucket"] = pd.qcut(
    signal3["gsc_sum_position"],
    q=4,
    duplicates="drop"
)

signal3_table = (
    signal3.groupby("bucket", observed=True)
    .agg(
        n=("gsc_sum_position", "size"),
        median=("gsc_sum_position", "median"),
        mean=("gsc_sum_position", "mean")
    )
    .reset_index()
)

display(signal3_table)

,bucket,n,median,mean
0,"(-0.001, 60.0]",7382235,0.0,2.963728
1,"(60.0, 481946.0]",2459143,340.0,1311.244630


### Verdict: CONFIRMED

The observed position buckets show meaningful variation in search position. This supports treating position as a useful contextual signal when reviewing search performance.

Because position is linked to the CTR-vs-position flag logic, this also provides a direct signal audit for the session's rule family.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test

The flag-linked signal tested here is GSC search position.

The session describes CTR-vs-position as part of the CTR-fix logic. I therefore checked whether the underlying position signal has enough observed variation to support its use as contextual evidence.

The result is descriptive: it tests whether the signal exists and varies in the data, not whether the FlyRank flag guarantees a successful action.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

flag_linked_check = march_df[
    [
        "gsc_sum_position",
        "gsc_impressions",
        "gsc_clicks"
    ]
].copy()

flag_linked_check["position_bucket"] = pd.qcut(
    flag_linked_check["gsc_sum_position"],
    q=4,
    duplicates="drop"
)

flag_linked_table = (
    flag_linked_check
    .groupby("position_bucket", observed=True)
    .agg(
        n=("gsc_sum_position", "size"),
        median_position=("gsc_sum_position", "median"),
        median_impressions=("gsc_impressions", "median"),
        median_clicks=("gsc_clicks", "median")
    )
    .reset_index()
)

display(flag_linked_table)

,position_bucket,n,median_position,median_impressions,median_clicks
0,"(-0.001, 60.0]",7382235,0.0,0.0,0.0
1,"(60.0, 481946.0]",2459143,340.0,35.0,0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

The signal audit shows how much observed variation exists in the search-performance fields used for refresh-oriented review.

The results support using these fields as directional decision-support signals rather than treating any single field as a guaranteed action trigger.

For a content team, the practical use is to combine observable search signals with human review instead of treating the baseline rule as an automatic publishing decision.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.